## **Car** Identifier

## Install ddgs package
 - Installs the ddgs package
 - Required to search DuckDuckGo for image urls


In [ ]:
!pip install -q ddgs

## Import fast.ai Packages

In [ ]:
from fastai.vision.all import *
from fastai.vision.widgets import *

## Import DDGS Class from ddgs and Utilities from Fastcore
 - Imports libraries needed to search Duckduckgo for images and creates a function to retrieve image URL



In [ ]:
from ddgs import DDGS
from fastcore.all import *

def search_images(keywords, max_images=200): return L(DDGS().images(keywords, max_results=max_images)).itemgot('image')
import time, json

## Search for Bugatti Image
 - Search duckduckgo for a bugatti photo and retrieve the first Url found


In [ ]:
urls = search_images('bugatti car photo', max_images = 1)
urls[0]

## Download the Bugatti Image
 - Import download_url function to download the image
 - Import path class to direct variable to look at a location to check if the image exists at that position or not
 - Done so that the image does not have to be downloaded again and again when executing the code

In [ ]:
from fastdownload import download_url
from pathlib import Path
dest = Path('bugatti.jpg')
if not dest.exists(): download_url(urls[0], dest, show_progress=False)


## Build an Image Dataset

- Create a tuple containing the image categories (`Bugatti` and `Koenigsegg`).
- Create a `Path` object representing the folder where the images will be stored.
- Iterate through each category using a `for` loop.
- Create a folder for each category if it does not already exist.
- Search DuckDuckGo for images of the current category and download them into the corresponding folder.
- Pause for 5 seconds to avoid sending too many requests to DuckDuckGo in a short time.
- Resize all downloaded images so their maximum dimension is 400 pixels.
- Repeat the same process for a few normal cars in a separate file

In [ ]:
searches = {'Bugatti supercar', 'Koenigsegg supercar'}
path = Path('bugatti_or_koenigsegg')

for o in searches:
  dest = (path/o)
  dest.mkdir(exist_ok=True, parents=True)
  download_images(dest, urls=search_images(f' {o} photo'))
  time.sleep(5)
  resize_images(path/o, max_size=400, dest=path/o)

other_searches={'Toyota Camry car', 'Honda Civic car', 'BMW 3 Series car'}
other_dest = path/'other car'
other_dest.mkdir(exist_ok=True, parents=True)

for term in other_searches:
  download_images(other_dest, urls=search_images(f' {term} photo'))
  time.sleep(5)

resize_images(path/'other car', max_size=400, dest=path/'other car')


# Delete Broken Images
 - Get image files scans the directory specified by path variable
 - Returns a list of image file paths found within it along with its subfolders
 - Verify images checks each image file for corruption or those that cannot be processed correctly
 - Returns a list of images which failed this verification which are assigned to failed variable
 - The map function applies a given function to every item in the list which is unlink, leading to deletion of the file from the system

In [ ]:
failed = verify_images(get_image_files(path))
failed.map(Path.unlink);

### Remove Duplicate Image Files


In [ ]:
fns = get_image_files(path)
dups = verify_images(fns)

from PIL import Image
import hashlib

def hash_image(fn):
    with Image.open(fn) as img:
        return hashlib.md5(img.tobytes()).hexdigest()

seen = {}
for fn in get_image_files(path):
    h = hash_image(fn)
    if h in seen:
        fn.unlink()
    else:
        seen[h] = fn

print(f'Remaining images: {len(get_image_files(path))}')

### Build Dataloader
 - Create a datablock that defines how the dataset should be constructed,we use image block for the input images and category block is for the image labels
 - We retrieve the image files from data set folder
 - We split the images such that 80 percent are used for training and 20 percent for validating the dataset
 - We use the name of the parent folder as labelling for the image
 - Resize all images to 192 x 192 pixels before training

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(path, bs=32)

dls.show_batch(max_n=6)

## Train and finetune the DataLoader
 - Use a pretrained resnet18 neural network as the base model
 - The model has already learned general image features from being trained through a large dataset
 - Finetune the model through our dataset of birds and forests images so it can classify images into the correct category

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

### Clean the Images in the Trained Model
 - Import ImageClassifierCleaner to clean the dls of images using the interactive widget

In [ ]:
from fastai.vision.widgets import ImageClassifierCleaner
cleaner = ImageClassifierCleaner(learn)
cleaner

### Delete the Filtered Images

In [ ]:
for idx in cleaner.delete(): cleaner.fns[idx].unlink()
for idx, cat in cleaner.change(): shutil.move(str(cleaner.fns[idx]), path/cat)

### Rebuild DataLoader

In [ ]:
dls = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=[Resize(192, method='squish')]
).dataloaders(path, bs=32)

dls.show_batch(max_n=6)

## Retrain and Finetune the DataLoader

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(3)

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()

## Checking whether the Model works or not
 - Use the previous bugatti or koenigsegg image which was downloaded
 - The finetuned model predicts whether the image belongs to the 'bugatti' , 'koenigsegg' or 'other car' category
 - The model calculates the probability of each category
 - Display the predicted class along with its probability

In [ ]:
from IPython.display import display

img = PILImage.create('bugatti.jpg')

print("Input Image:")
display(img.to_thumb(300,300))

pred, pred_idx, probs = learn.predict(img)

print("\nPrediction:")
print(f"Predicted class: {pred}")
print(f"Bugatti: {probs[0]*100:.2f}%")
print(f"Koenigsegg: {probs[1]*100:.2f}%")
print(f"Other Car: {probs[2]*100:.2f}%")